# Reproduction: BGE-M3 on the course testbed

Pair 11, Oliver Wakeford and Mathis Lesgourgues. Paper: Chen et al., *M3-Embedding* (Findings of ACL 2024), model `BAAI/bge-m3`.

The paper's central claims we test at small scale:
1. **Multi-functionality** (Table 1): one model gives three scores, and their fusion beats each one alone: Sparse < Dense < Multi-vec ≈ Dense+Sparse < All.
2. **Multi-linguality** (Table 2): with French queries against English passages, Dense, Multi-vec and All lose almost nothing, Sparse loses a lot.

The paper uses MIRACL (18 languages) and MKQA (25 languages, about 1M passages). We use the three testbed datasets (English, 3.6k to 8.7k documents) and the 300 SciFact test queries machine-translated to French. So only the *ordering* of the modes and the *size of the loss* can be compared, not the absolute numbers.

This notebook encodes nothing and runs in a few seconds. It reads the files written by `method.py`, `crosslingual.py` and `tune_weights.py` (see the README).

In [1]:
import json
import sys

import numpy as np
import pandas as pd

sys.path.insert(0, "../..")                       # repo root, for testbed.py
import testbed

pd.set_option("display.precision", 3)
MODES = ["dense", "sparse", "multi-vec", "dense+sparse", "all"]
LABEL = {"dense": "Dense", "sparse": "Sparse", "multi-vec": "Multi-vec", "dense+sparse": "Dense+Sparse", "all": "All"}
FILE = {m: f"results{'' if m == 'all' else '_' + m.replace('+', '_')}.json" for m in MODES}
results = {m: json.load(open(FILE[m]))["datasets"] for m in MODES}
per_query = json.load(open("results_per_query.json"))

## 1. The five modes (paper Table 1, scaled down)

nDCG@10 × 100. The paper's columns are MIRACL averages and its English column; ours are the testbed datasets.

In [2]:
# paper, Table 1: MIRACL dev, nDCG@10 (average over 18 languages, and English)
paper_miracl = {"avg": dict(zip(MODES, [69.2, 53.9, 70.5, 70.4, 71.5])),
                "en": dict(zip(MODES, [56.9, 43.8, 59.3, 58.8, 59.6]))}
rows = []
for m in MODES:
    row = {"mode": LABEL[m], "paper MIRACL avg": paper_miracl["avg"][m], "paper MIRACL en": paper_miracl["en"][m]}
    row |= {name: 100 * results[m][name]["ndcg10"] for name in testbed.DATASETS}
    rows.append(row)
ablation = pd.DataFrame(rows).set_index("mode")
ablation.round(1)

,paper MIRACL avg,paper MIRACL en,scifact,nfcorpus,arguana
mode,,,,,
Dense,69.2,56.9,63.3,31.5,54.3
Sparse,53.9,43.8,63.7,28.2,38.7
Multi-vec,70.5,59.3,68.3,33.4,51.2
Dense+Sparse,70.4,58.8,66.7,32.6,52.4
All,71.5,59.6,69.2,33.6,52.7


Same ordering as the paper on SciFact and NFCorpus: All is best, then Multi-vec, then Dense+Sparse. Two things differ:

* **Sparse is much closer to Dense here.** On SciFact it is even level with Dense (63.7 vs 63.3), where the paper has it 15 points below on MIRACL. SciFact claims share many words with their abstracts, which helps any lexical method (BM25 alone gets 0.676 there).
* **ArguAna goes the other way.** Dense is the best mode (54.3), and both Multi-vec (51.2) and All (52.7) are below it. ArguAna's queries are whole arguments and the target is the counter-argument, so token-level matching rewards the documents that *agree* with the query.

Each step of the ablation, with a paired t-test over queries. 15 tests, so with Bonferroni a p-value must be below 0.05 / 15 = 0.0033.

In [3]:
# each step of the paper's ablation, as a paired t-test over the queries (nDCG@10)
steps = [("multi-vec", "dense", "Multi-vec reranking over Dense"),
         ("dense+sparse", "dense", "adding Sparse to Dense"),
         ("all", "multi-vec", "All over Multi-vec"),
         ("all", "dense+sparse", "All over Dense+Sparse"),
         ("dense", "sparse", "Dense over Sparse")]
rows = []
for a, b, what in steps:
    for name in testbed.DATASETS:
        c = testbed.compare(np.array(per_query[name][a]["ndcg10"]), np.array(per_query[name][b]["ndcg10"]))
        rows.append({"step": what, "dataset": name, "delta": c["delta"], "ci95": c["ci95"], "p": c["p"]})
tests = pd.DataFrame(rows)
tests["significant (Bonferroni, 15 tests)"] = tests["p"] < 0.05 / len(tests)
tests

,step,dataset,delta,ci95,p,"significant (Bonferroni, 15 tests)"
0,Multi-vec reranking over Dense,scifact,0.050,"[0.029, 0.0726]",1.198e-05,True
1,Multi-vec reranking over Dense,nfcorpus,0.019,"[0.0076, 0.0315]",1.318e-03,True
2,Multi-vec reranking over Dense,arguana,-0.030,"[-0.0398, -0.0207]",1.171e-09,True
3,adding Sparse to Dense,scifact,0.034,"[0.0211, 0.0484]",1.495e-06,True
4,adding Sparse to Dense,nfcorpus,0.011,"[0.0051, 0.0178]",4.272e-04,True
5,adding Sparse to Dense,arguana,-0.018,"[-0.0278, -0.0084]",2.550e-04,True
6,All over Multi-vec,scifact,0.009,"[-0.004, 0.0215]",1.900e-01,False
7,All over Multi-vec,nfcorpus,0.002,"[-0.0039, 0.0085]",4.651e-01,False
8,All over Multi-vec,arguana,0.015,"[0.0066, 0.0229]",3.160e-04,True
9,All over Dense+Sparse,scifact,0.024,"[0.0115, 0.0376]",3.954e-04,True


* Multi-vec reranking and adding Sparse to Dense both help on SciFact and NFCorpus, and both hurt on ArguAna. All six differences survive Bonferroni.
* All over Multi-vec: +0.009 and +0.002 on SciFact and NFCorpus, both noise (p = 0.19 and 0.47). Only on ArguAna does the fusion add something on top of Multi-vec (+0.015), and there both are below Dense.
* So at this scale the gain of All comes from Multi-vec. The paper reports All 1.0 point above Multi-vec on the MIRACL average, which our datasets are too small to resolve.

## 2. The testbed row

`results.json` is mode All against RRF (BM25 + MiniLM), the spring pipeline the paper menu asks us to compare with. It is not the strongest baseline everywhere: the cross-encoder is higher on NFCorpus and bge-small-en-v1.5, the closest dense baseline, on ArguAna. So the second table compares every mode with every baseline.

In [4]:
rows = []
for name in testbed.DATASETS:
    r = results["all"][name]
    c = r["ndcg10_vs_baseline"]
    rows.append({"dataset": name, "BGE-M3 All": r["ndcg10"], "RRF": r["baseline"]["ndcg10"], "delta": c["delta"],
                 "ci95": c["ci95"], "p": c["p"], "Recall@100": r["recall100"], "RRF Recall@100": r["baseline"]["recall100"]})
pd.DataFrame(rows).set_index("dataset")

,BGE-M3 All,RRF,delta,ci95,p,Recall@100,RRF Recall@100
dataset,,,,,,,
scifact,0.692,0.718,-0.026,"[-0.0544, 0.0027]",0.078,0.914,0.948
nfcorpus,0.336,0.342,-0.006,"[-0.0218, 0.0104]",0.470,0.289,0.310
arguana,0.527,0.549,-0.022,"[-0.0361, -0.0078]",0.002,0.980,0.985


In [5]:
# every mode against every baseline in the repo, nDCG@10 delta (p)
base = testbed.stored()
rows = []
for name in testbed.DATASETS:
    for b in testbed.SYSTEMS:
        row = {"dataset": name, "baseline": b, "baseline nDCG@10": np.mean(base[name][b]["ndcg10"])}
        for m in MODES:
            c = testbed.compare(np.array(per_query[name][m]["ndcg10"]), np.array(base[name][b]["ndcg10"]))
            row[LABEL[m]] = f"{c['delta']:+.3f} ({c['p']:.2g})"
        rows.append(row)
pd.DataFrame(rows).set_index(["dataset", "baseline"])

baseline nDCG@10             Dense            Sparse  \
dataset  baseline                                                         
scifact  bm25                 0.676    -0.043 (0.014)    -0.039 (0.014)   
         minilm               0.646     -0.012 (0.43)     -0.009 (0.66)   
         bge                  0.710  -0.077 (7.2e-08)  -0.073 (5.4e-05)   
         rrf                  0.718  -0.085 (3.2e-07)  -0.081 (2.9e-06)   
         ce                   0.688  -0.055 (0.00023)   -0.051 (0.0024)   
nfcorpus bm25                 0.321      -0.006 (0.5)  -0.039 (3.1e-05)   
         minilm               0.316     -0.002 (0.86)   -0.034 (0.0029)   
         bge                  0.340   -0.026 (0.0054)  -0.058 (2.7e-07)   
         rrf                  0.342   -0.028 (0.0017)  -0.060 (2.8e-09)   
         ce                   0.347  -0.033 (0.00015)  -0.065 (2.4e-12)   
arguana  bm25                 0.516   +0.027 (0.0021)  -0.129 (4.3e-55)   
         minilm               0.503  +0.040 (1.3e-06)  -0.116 (7.2e-32)   
         bge                  0.606  -0.063 (6.9e-16)    -0.219 (2e-98)   
         rrf                  0.549     -0.006 (0.41)    -0.162 (6e-76)   
         ce                   0.422  +0.120 (1.3e-33)  -0.036 (0.00021)   

                          Multi-vec      Dense+Sparse               All  
dataset  baseline                                                        
scifact  bm25         +0.007 (0.66)      -0.009 (0.6)     +0.015 (0.33)  
         minilm      +0.038 (0.025)     +0.022 (0.16)   +0.046 (0.0046)  
         bge         -0.027 (0.054)   -0.043 (0.0017)     -0.018 (0.16)  
         rrf          -0.035 (0.02)   -0.050 (0.0011)    -0.026 (0.078)  
         ce           -0.005 (0.71)     -0.021 (0.12)     +0.004 (0.78)  
nfcorpus bm25         +0.013 (0.11)     +0.005 (0.54)    +0.016 (0.057)  
         minilm       +0.018 (0.07)     +0.010 (0.29)    +0.020 (0.033)  
         bge          -0.006 (0.49)    -0.014 (0.098)     -0.004 (0.65)  
         rrf          -0.008 (0.34)    -0.016 (0.049)     -0.006 (0.47)  
         ce          -0.013 (0.085)   -0.021 (0.0077)     -0.011 (0.14)  
arguana  bm25         -0.003 (0.69)     +0.009 (0.27)     +0.011 (0.15)  
         minilm       +0.010 (0.24)   +0.022 (0.0093)   +0.025 (0.0027)  
         bge       -0.093 (2.5e-29)  -0.082 (4.6e-24)  -0.079 (7.4e-23)  
         rrf       -0.037 (1.5e-06)  -0.025 (0.00066)   -0.022 (0.0024)  
         ce        +0.090 (9.8e-21)  +0.102 (3.4e-27)  +0.105 (1.6e-28)

BGE-M3 All does not beat the spring pipeline. On SciFact (−0.026, p = 0.078) and NFCorpus (−0.006, p = 0.47) the difference is noise; on ArguAna it loses (−0.022, p = 0.002). Recall@100 is lower on all three.

Against bge-small-en-v1.5 (33M parameters, English only) the picture is the same on SciFact and NFCorpus (All −0.018, p = 0.16 and −0.004, p = 0.65), but on ArguAna every BGE-M3 mode loses clearly (All −0.079, Dense −0.063). We have not tested why bge-small is so strong there (0.606, the best baseline on ArguAna by 0.06). A 568M multilingual encoder is at best level with small English models on English data: its advantage is in the languages, which is section 3.

## 3. Cross-lingual (paper Table 2, scaled down)

The 300 SciFact test queries translated to French with NLLB-200 (`translate_scifact.py`), against the English corpus. The paper's metric on MKQA is Recall@100; we also give nDCG@10.

In [6]:
# paper, Table 2: MKQA, Recall@100, French queries against English passages
paper_mkqa_fr = dict(zip(MODES, [76.2, 53.9, 76.4, 76.6, 76.6]))
cross = json.load(open("results_crosslingual.json"))
rows = []
for m in MODES:
    fr = cross[m]["fr"]
    rows.append({"mode": LABEL[m], "paper MKQA fr R@100": paper_mkqa_fr[m],
                 "R@100 en": 100 * cross[m]["en"]["recall100"], "R@100 fr": 100 * fr["recall100"],
                 "p (R@100)": fr["recall100_vs_en"]["p"],
                 "nDCG@10 en": 100 * cross[m]["en"]["ndcg10"], "nDCG@10 fr": 100 * fr["ndcg10"],
                 "delta nDCG@10": 100 * fr["ndcg10_vs_en"]["delta"], "p (nDCG@10)": fr["ndcg10_vs_en"]["p"]})
pd.DataFrame(rows).set_index("mode").round({"paper MKQA fr R@100": 1, "R@100 en": 1, "R@100 fr": 1,
                                            "nDCG@10 en": 1, "nDCG@10 fr": 1, "delta nDCG@10": 1})

,paper MKQA fr R@100,R@100 en,R@100 fr,p (R@100),nDCG@10 en,nDCG@10 fr,delta nDCG@10,p (nDCG@10)
mode,,,,,,,,
Dense,76.2,90.4,89.8,4.607e-01,63.3,58.6,-4.7,1.059e-05
Sparse,53.9,89.4,75.8,4.964e-08,63.7,44.0,-19.7,1.672e-18
Multi-vec,76.4,91.0,90.4,5.568e-01,68.3,63.4,-4.9,2.224e-05
Dense+Sparse,76.6,91.7,91.4,7.395e-01,66.7,60.6,-6.2,8.376e-08
All,76.6,91.4,91.5,9.172e-01,69.2,63.4,-5.7,1.763e-06


At the paper's metric the result reproduces: Dense, Multi-vec, Dense+Sparse and All lose nothing measurable on Recall@100 (all p > 0.4), and Sparse loses 13.6 points (p < 10⁻⁷). Sparse can only score tokens the French query shares with the English abstract, so what survives is names, acronyms and numbers.

nDCG@10 is stricter: the four other modes lose 4.7 to 6.2 points (All −5.7, p < 10⁻⁵) and Sparse loses 19.7, so French queries still find the abstract in the top 100 but rank it lower. These are 10 more tests (5 modes × 2 metrics); with Bonferroni over them (0.05 / 10) the same differences stay significant and the same ones stay noise.

In [7]:
# Sparse's loss relative to the best mode, ours and the paper's (in points of Recall@100)
best_ours, best_paper = max(100 * cross[m]["fr"]["recall100"] for m in MODES), max(paper_mkqa_fr.values())
print(f"Sparse vs best mode, French queries: ours {100 * cross['sparse']['fr']['recall100'] - best_ours:+.1f}, "
      f"paper {paper_mkqa_fr['sparse'] - best_paper:+.1f}")
c = cross["all"]["fr_checked"]
print(f"All, the {c['n_queries']} hand-checked queries: nDCG@10 delta {c['ndcg10_vs_en']['delta']:+.3f}, p {c['ndcg10_vs_en']['p']:.2f}")

Sparse vs best mode, French queries: ours -15.7, paper -22.7
All, the 26 hand-checked queries: nDCG@10 delta -0.008, p 0.68


Sparse falls 15.7 points behind the best mode with French queries; the paper's gap is 22.7. On the 26 hand-checked queries that were translated correctly (`check_30.csv`), All loses only 0.008 (p = 0.68). That subset is not random: it was picked for having the most acronyms and numbers, which match in both languages, so it does not show that the whole loss is translation error (see `error_analysis.md`).

## 4. Fusion weights

The paper's weights (1, 0.3, 1) come from MIRACL. We tuned w_lex and w_mul on the 809 SciFact **training** queries only, then scored the 300 test queries once with each set.

In [8]:
w = json.load(open("results_weights.json"))
grid = pd.DataFrame(w["grid"]).pivot(index="w_lex", columns="w_mul", values="train_ndcg10")
print(f"{w['n_train_queries']} training queries. Paper weights (1, {w['paper']['w_lex']}, {w['paper']['w_mul']}): "
      f"test nDCG@10 {w['paper']['test_ndcg10']:.3f}. Tuned (1, {w['tuned']['w_lex']}, {w['tuned']['w_mul']}): "
      f"test nDCG@10 {w['tuned']['test_ndcg10']:.3f}")
c = w["tuned_vs_paper"]["ndcg10"]
print(f"tuned - paper: {c['delta']:+.3f} {c['ci95']}, p {c['p']:.2f}")
grid

809 training queries. Paper weights (1, 0.3, 1.0): test nDCG@10 0.692. Tuned (1, 0.1, 1.5): test nDCG@10 0.689
tuned - paper: -0.002 [-0.0094, 0.0046], p 0.53


w_mul,0.0,0.5,1.0,1.5,2.0
w_lex,,,,,
0.0,0.704,0.727,0.728,0.731,0.730
0.1,0.716,0.730,0.732,0.734,0.732
0.2,0.722,0.730,0.733,0.734,0.733
0.3,0.725,0.731,0.733,0.734,0.734
0.5,0.727,0.730,0.731,0.733,0.733
1.0,0.717,0.729,0.733,0.733,0.733


The training grid is flat: every w_mul ≥ 0.5 is within 0.007 of the best. The tuned weights (1, 0.1, 1.5) give 0.689 on test against 0.692 for the paper's, a difference of −0.002 (p = 0.53), which is noise. The paper's weights transfer, and they stay the default in `method.py`.

## 5. Compute

Everything ran on one Mac (Apple Silicon, PyTorch MPS, fp16). Encoding with all three heads took, as printed by `method.py` on its first run, 42 s for NFCorpus (3,633 docs), 66 s for SciFact (5,183) and 67 s for ArguAna (8,674), plus 15 s for the 1,406 ArguAna queries. `method.py` caches the encodings, so all five modes, the cross-lingual runs and the weight grid reuse one encoding per corpus; with the cache, the whole pipeline runs in about 2 minutes. A second run from a clean clone printed 46 s and 70 s for NFCorpus and ArguAna, and gave the same results files byte for byte. The same numbers came out of Mathis's earlier run to within 0.0012 nDCG@10.